In [45]:
import pandas as pd

df = pd.read_csv('../data/results.csv')

print(df.shape)
df.head(10)

(47126, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


In [46]:
# See all the different tournament types
print(df['tournament'].value_counts().head(20))

# Filter to only competitive matches (no friendlies)
competitive = df[df['tournament'] != 'Friendly']

# Filter to just World Cup matches
world_cup = df[df['tournament'] == 'FIFA World Cup']

print(f"\nTotal competitive matches: {len(competitive)}")
print(f"Total World Cup matches: {len(world_cup)}")
print(f"\nWorld Cup matches sample:")
world_cup.tail(10)

tournament
Friendly                                17902
FIFA World Cup qualification             8052
UEFA Euro qualification                  2824
African Cup of Nations qualification     2116
FIFA World Cup                            964
Copa América                              841
African Cup of Nations                    793
AFC Asian Cup qualification               764
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        595
British Home Championship                 517
UEFA Nations League                       474
AFC Asian Cup                             421
Gulf Cup                                  395
Gold Cup                                  389
UEFA Euro                                 388
Island Games                              379
Asian Games                               368
COSAFA Cup                                354
Name: count, dtype: int64

Total competitive matches: 29224
Total Wor

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
45683,2022-12-06,Morocco,Spain,0.0,0.0,FIFA World Cup,Al Rayyan,Qatar,True
45684,2022-12-06,Portugal,Switzerland,6.0,1.0,FIFA World Cup,Lusail,Qatar,True
45685,2022-12-09,Croatia,Brazil,1.0,1.0,FIFA World Cup,Al Rayyan,Qatar,True
45686,2022-12-09,Netherlands,Argentina,2.0,2.0,FIFA World Cup,Lusail,Qatar,True
45688,2022-12-10,Morocco,Portugal,1.0,0.0,FIFA World Cup,Doha,Qatar,True
45689,2022-12-10,England,France,1.0,2.0,FIFA World Cup,Al Khor,Qatar,True
45691,2022-12-13,Argentina,Croatia,3.0,0.0,FIFA World Cup,Lusail,Qatar,True
45692,2022-12-14,France,Morocco,2.0,0.0,FIFA World Cup,Al Khor,Qatar,True
45696,2022-12-17,Croatia,Morocco,2.0,1.0,FIFA World Cup,Al Rayyan,Qatar,True
45698,2022-12-18,Argentina,France,3.0,3.0,FIFA World Cup,Lusail,Qatar,True


In [47]:
# Add a result column from home team's perspective
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

df['result'] = df.apply(get_result, axis=1)

# Function to get a team's last N results before a given date
def get_recent_form(team, date, df, n=10):
    past = df[((df['home_team'] == team) | (df['away_team'] == team)) & (df['date'] < date)]
    past = past.sort_values('date').tail(n)
    
    wins = 0
    for _, row in past.iterrows():
        if (row['home_team'] == team and row['result'] == 'home_win') or \
           (row['away_team'] == team and row['result'] == 'away_win'):
            wins += 1
    return wins / len(past) if len(past) > 0 else 0

# Test it — how many of Brazil's last 5 games before the 2022 WC final did they win?
print("Brazil recent form:", get_recent_form('Brazil', '2022-12-18', df))
print("Argentina recent form:", get_recent_form('Argentina', '2022-12-18', df))

Brazil recent form: 0.6
Argentina recent form: 0.8


In [48]:
df['result'] = None
df.head(3)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,None
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,None
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,None


In [49]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'
df['result'] = df.apply(get_result, axis=1)

In [50]:
df[['home_team', 'away_team', 'home_score', 'away_score', 'result']].head(10)

,home_team,away_team,home_score,away_score,result
0,Scotland,England,0.0,0.0,draw
1,England,Scotland,4.0,2.0,home_win
2,Scotland,England,2.0,1.0,home_win
3,England,Scotland,2.0,2.0,draw
4,Scotland,England,3.0,0.0,home_win
5,Scotland,Wales,4.0,0.0,home_win
6,England,Scotland,1.0,3.0,away_win
7,Wales,Scotland,0.0,2.0,away_win
8,Scotland,England,7.0,2.0,home_win
9,Scotland,Wales,9.0,0.0,home_win


In [51]:
def get_recent_form(team, date, n=10):
    # Find all matches where this team played
    team_matches = df[
        (df['home_team'] == team) | (df['away_team'] == team)
        ]
    
    # Only keep matches BEFORE the given date
    team_matches = team_matches[team_matches['date'] < date]

    # Take the last n matches
    team_matches = team_matches.sort_values('date').tail(n)

    # Count the wins
    wins = 0
    for _, row in team_matches.iterrows():
        if (row['home_team'] == team and row['result'] == 'home_win' or \
            row['away_team'] == team and row['result'] == 'away_win'):
            wins += 1

    # Return the win rate 
    return wins/ len(team_matches) if len(team_matches) > 0 else 0

In [52]:
print(get_recent_form('Brazil', '2022-12-18'))
print(get_recent_form('Argentina', '2022-12-18'))

0.6
0.8


In [53]:
def get_avg_goals_scored(team, date, n=10):
    # Find all matches where this team played
    team_matches = df[
        (df['home_team'] == team) | (df['away_team'] == team)
        ]
    
    # Only keep matches BEFORE the given date
    team_matches = team_matches[team_matches['date'] < date]

    # Take the last n matches
    team_matches = team_matches.sort_values('date').tail(n)

    # Add up all the goals scored
    goals = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            goals += row['home_score']
        else:
            goals += row ['away_score']

    # Return the average
    return goals / n if len(team_matches) > 0 else 0

In [54]:
print(get_avg_goals_scored('Brazil', '2022-12-18'))
print(get_avg_goals_scored('Argentina', '2022-12-18'))

1.6
2.2


In [55]:
def get_avg_goals_conceded(team, date, n=10):
    # Find all matches where this team played
    team_matches = df[
        (df['home_team'] == team) | (df['away_team'] == team)
        ]
    
    # Only keep matches BEFORE the given date
    team_matches = team_matches[team_matches['date'] < date]

    # Take the last n matches
    team_matches = team_matches.sort_values('date').tail(n)

    # Add up all the goals conceded
    conceded = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            conceded += row['away_score']
        else:
            conceded += row ['home_score']

    # Return the average
    return conceded / n if len(team_matches) > 0 else 0


In [56]:
print(get_avg_goals_conceded('Brazil', '2022-12-18'))
print(get_avg_goals_conceded('Argentina', '2022-12-18'))

0.6
0.6


In [57]:
def get_head_to_head(team1, team2, date):
    h2h = df[
        ((df['home_team'] == team1) & (df['away_team'] == team2)) |
        ((df['home_team'] == team2) & (df['away_team'] == team1))
    ]
    
    h2h = h2h[h2h['date'] < date]
    
    
    wins = 0
    for _, row in h2h.iterrows():
        if ((row['home_team'] == team1 and row['result'] == 'home_win') or
            (row['away_team'] == team1 and row['result'] == 'away_win')):
            wins += 1
    
    
    return round(wins / len(h2h), 2) if len(h2h) > 0 else 0

print(get_head_to_head('Brazil', 'Argentina', '2022-12-18'))
    

0.4


In [58]:
print(get_recent_form('Brazil', '2022-12-18'))
print(get_avg_goals_scored('Brazil', '2022-12-18'))
print(get_avg_goals_conceded('Brazil', '2022-12-18'))
print(get_head_to_head('Brazil', 'Argentina', '2022-12-18'))

0.6
1.6
0.6
0.4


In [90]:
world_cup = df[df['tournament'] == 'FIFA World Cup'].copy()

rows = []

for _, match in world_cup.iterrows():
    home = match['home_team']
    away = match['away_team']
    date = match['date']

    row = {
        'home_form': get_recent_form(home, date),
        'home_goals_scored': get_avg_goals_scored(home, date),
        'home_goals_conceded': get_avg_goals_conceded(home, date),
        'home_h2h': get_head_to_head(home, away, date),
        'away_form': get_recent_form(away, date),
        'away_goals_scored': get_avg_goals_scored(away, date),
        'away_goals_conceded': get_avg_goals_conceded(away, date),
        'away_h2h': get_head_to_head(away, home, date),
        'home_ranking': get_fifa_ranking(home, date, rankings),
        'away_ranking': get_fifa_ranking(away, date, rankings),
        'result': match['result']
}
    rows.append(row)
training_data['ranking_diff'] = training_data['home_ranking'] - training_data['away_ranking']
training_data = pd.DataFrame(rows)
print(training_data.shape)
training_data.head()


    

(964, 11)


,home_form,home_goals_scored,home_goals_conceded,home_h2h,away_form,away_goals_scored,away_goals_conceded,away_h2h,home_ranking,away_ranking,result
0,0.4,1.8,1.8,0.0,0.4,3.4,3.6,0.0,0.0,0.0,away_win
1,0.2,1.6,3.0,0.0,0.4,1.8,2.8,0.0,0.0,0.0,home_win
2,0.4,2.2,2.2,0.0,0.4,2.8,2.0,0.0,0.0,0.0,away_win
3,0.2,1.0,3.8,0.0,0.6,3.2,1.8,0.0,0.0,0.0,away_win
4,0.6,2.0,0.4,0.0,0.4,1.8,2.6,0.0,0.0,0.0,home_win


In [91]:
X = training_data.drop(columns=['result'])
y = training_data['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {round(accuracy * 100, 2)}%")

Model accuracy: 50.26%


In [68]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print(f"Training matches: {len(X_train)}")
print(f"Testing matches: {len(X_test)}")


Training matches: 771
Testing matches: 193


In [70]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators = 100, random_state= 42)
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [74]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model accuracy: {round(accuracy * 100, 2)}%")

Model accuracy: 49.74%


In [75]:
import pandas as pd

feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)


home_goals_scored      0.156695
away_goals_scored      0.156295
away_goals_conceded    0.146483
home_goals_conceded    0.144434
home_h2h               0.111360
away_h2h               0.110523
away_form              0.088010
home_form              0.086200
dtype: float64


In [76]:
rankings = pd.read_csv('../data/fifa_rankings.csv')
print(rankings.shape)
rankings.head()

(69992, 9)


,Unnamed: 0,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [80]:
# Get unique team names from results
result_teams = set(df['home_team'].unique())

# Get unique team names from rankings
ranking_teams = set(rankings['country_full'].unique())

# Find teams in results that don't appear in rankings
missing = result_teams - ranking_teams
missing_clean = [t for t in missing if isinstance(t, str)]
print(f"Teams with no ranking match: {len(missing_clean)}")
print(missing_clean)

Teams with no ranking match: 133
['Saint Barthélemy', 'Barawa', 'Mayotte', 'Mapuche', 'Catalonia', 'Chechnya', 'Felvidék', 'Bonaire', 'United Koreans in Japan', 'South Korea', 'Vietnam Republic', 'Parishes of Jersey', 'Iran', 'Matabeleland', 'Silesia', 'Réunion', 'North Korea', 'North Vietnam', 'Greenland', 'Saint Pierre and Miquelon', 'Alderney', 'Saugeais', 'Kernow', 'Yemen DPR', 'Chagos Islands', 'Padania', 'Western Isles', 'Monaco', 'County of Nice', 'Western Armenia', 'Northern Mariana Islands', 'Luhansk PR', 'Orkney', 'Micronesia', 'Palau', 'Yorkshire', 'Hmong', 'Saarland', 'Cascadia', 'Kárpátalja', 'Saint Helena', 'Åland Islands', 'Åland', 'Malaya', 'Basque Country', 'South Ossetia', 'Galicia', 'DR Congo', 'East Timor', 'Shetland', 'Saint Martin', 'Tuvalu', 'Sealand', 'Corsica', 'Occitania', 'Wallis Islands and Futuna', 'Northern Cyprus', 'Sint Maarten', 'Székely Land', 'Western Australia', 'Abkhazia', 'French Guiana', 'Délvidék', 'Sark', 'Panjab', 'United States', 'Romani peopl

In [81]:
wc_teams = set(world_cup['home_team'].unique()) | set(world_cup['away_team'].unique())
wc_missing = wc_teams - ranking_teams
print(f"World Cup teams with no ranking match: {len(wc_missing)}")
print(wc_missing)

World Cup teams with no ranking match: 9
{'Iran', 'Turkey', 'United States', 'North Korea', 'Ivory Coast', 'German DR', 'South Korea', 'DR Congo', 'Czech Republic'}


In [82]:
name_mapping = {
    'Iran': 'IR Iran',
    'Turkey': 'Türkiye',
    'United States': 'USA',
    'North Korea': "Korea DPR",
    'Ivory Coast': "Côte d'Ivoire",
    'German DR': 'Germany',
    'South Korea': 'Korea Republic',
    'DR Congo': 'Congo DR',
    'Czech Republic': 'Czechia'
}

In [84]:
def get_fifa_ranking(team, date, rankings):
    # Use mapped name if it exists

    lookup_name = name_mapping.get(team, team)

    # Get all rankings for this team before the certain date

    team_rankings = rankings[
        (rankings['country_full'] == lookup_name) &
        (rankings['rank_date'] < date)
        ]
    if len(team_rankings) == 0:
        return 0

    # Return the most recent ranking points

    return team_rankings.sort_values('rank_date').iloc[-1]['total_points']

print(get_fifa_ranking('Brazil', '2022-12-18', rankings))
print(get_fifa_ranking('Argentina', '2022-12-18', rankings))

1841.3
1773.88


In [87]:
print(training_data.columns.tolist())

['home_form', 'home_goals_scored', 'home_goals_conceded', 'home_h2h', 'away_form', 'away_goals_scored', 'away_goals_conceded', 'away_h2h', 'home_ranking', 'away_ranking', 'result']


In [89]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)

home_goals_scored      0.132347
away_goals_scored      0.127005
away_goals_conceded    0.117841
home_goals_conceded    0.117248
away_ranking           0.097576
home_h2h               0.091830
home_ranking           0.089740
away_h2h               0.087904
away_form              0.069552
home_form              0.068955
dtype: float64
